# Lab 0 · Environment & Sanity Check

**~20 minutes.** Everyone gets to the same green banner before anyone moves on.

By the end of this notebook you will know what GPU you have, what numeric
precision it can do, and whether your Hugging Face access works.

### Before you run anything

In the right-hand panel:

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** |
| Persistence | Off |

Both need phone verification on your Kaggle account. If you can't select
them, that's why.

## 1.2 What GPU did we get?

`nvidia-smi` is the first command to reach for on any GPU machine. Read
the memory column — that number is the ceiling on everything you do today.

In [ ]:
!nvidia-smi

## 1.4 Install the stack

Four packages matter:

- **unsloth** — the fast fine-tuning layer; patches the model for ~2× speed and ~50% less VRAM
- **peft** — the LoRA implementation (Parameter-Efficient Fine-Tuning)
- **trl** — `SFTTrainer`, the supervised fine-tuning loop
- **bitsandbytes** — the 4-bit quantization kernels that make QLoRA possible

`--no-deps` is deliberate: it stops pip replacing Kaggle's torch build with
an incompatible one. **If the install fails**, see `docs/troubleshooting.md`
— pinned fallback versions are listed there.

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

## Locate the repo and load the shared config

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-USERNAME/LLM-lab.git"   # TODO: your repo

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

## 1.3 Compute capability decides your precision

Every NVIDIA GPU has a *compute capability* — `sm75`, `sm80`, `sm90`. It
determines which numeric formats the silicon supports natively.

| | fp16 | bf16 | Flash Attention 2 |
|---|---|---|---|
| T4 (sm75, 2018) | yes | **no** | **no** |
| A100 (sm80, 2020) | yes | yes | yes |
| H100 (sm90, 2022) | yes | yes | yes |

**bf16** has the same exponent range as fp32 but fewer mantissa bits. That
range is what stops training from overflowing, so bf16 is more forgiving
than fp16 — but the T4 predates it.

On a T4 we use fp16 with loss scaling instead. It works fine. You'll see
`bfloat16: no` below and that is **correct, not an error**.

> ↳ Slide: *Model Size based on Parameter Count and Precision*

In [ ]:
import gpu_check
info = gpu_check.report()

## 1.5 Hugging Face authentication

Llama 3.2 is a **gated** model: you must accept Meta's licence and be
logged in. If your approval hasn't come through, don't wait — set
`USE_UNGATED_MODEL = True` and use Qwen2.5-3B instead. Every lab works
identically.

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

## 1.6 Disk reality check

`/kaggle/working` holds about **20 GB**, and Lab 5 will merge a 6.4 GB
model and then convert it to GGUF. That is tight. We plan for it there;
just note the number now.

In [ ]:
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working:  {free/2**30:5.1f} GB free of {total/2**30:.1f} GB")
print(f"\nLab 5 needs roughly:")
print(f"  merged 16-bit model    6.4 GB")
print(f"  GGUF q4_k_m            2.0 GB")
print(f"  base model cache       2.3 GB")
print(f"  ---------------------------")
print(f"  total                 10.7 GB  ->  {'OK' if free/2**30 > 12 else 'TIGHT'}")

## 1.7 Green light

If this prints `READY`, you're set. If not, the message says what to fix.

In [ ]:
checks = {
    "GPU available":      info.available,
    "Repo on sys.path":   True,
    "HF authenticated":   bool(os.environ.get("HF_TOKEN")) or config.USE_UNGATED_MODEL,
    "Disk > 12 GB free":  shutil.disk_usage("/kaggle/working")[2] / 2**30 > 12,
}

width = max(len(k) for k in checks)
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name:<{width}}")

print()
if all(checks.values()):
    print("  READY — continue to Lab 1.")
else:
    print("  NOT READY — see docs/troubleshooting.md, or ask an instructor.")
    print("  Note: 'HF authenticated' can be skipped by setting")
    print("        USE_UNGATED_MODEL = True in common/config.py")

---

### Next: `01_baseline_inference.ipynb` — meet the model, and watch it invent answers

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.